# 03 – AQI Data Diagnosis

1. Understand the station-hour data
→ structure, timestamps, duplicates, station coverage

2. Diagnose pollutant missingness
→ missing percentages, station-level availability, long gaps, zero-availability combinations

3. Examine the existing AQI
→ missingness, distribution, extreme values

4. Reproduce the project's AQI methodology
→ rolling windows → pollutant sub-indices → final AQI

5. Validate the reproduction
→ 99.94% agreement with the dataset's existing AQI

6. Determine whether missing AQI can be recovered
→ 0 of 570,190 missing AQI rows can be calculated under this methodology

Data diagnosis + AQI validation notebook.

## 1. Import Libraries

In [ ]:
import sys
from pathlib import Path

# Add the project root to sys.path so 'src' is importable
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np

## 2. Load Dataset

In [ ]:
from src.data_loader import load_station_hour

df_station_hour = load_station_hour()

print("Shape:", df_station_hour.shape)
print(df_station_hour.head())

c:\Users\hp\Desktop\Internship_AQI_Prediction\src\data_loader.py:15: DtypeWarning: Columns (15: AQI_Bucket) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(file_path)


Shape: (2589083, 16)
  StationId             Datetime  PM2.5    PM10    NO    NO2    NOx    NH3  \
0     AP001  2017-11-24 17:00:00  60.50   98.00  2.35  30.80  18.25   8.50   
1     AP001  2017-11-24 18:00:00  65.50  111.25  2.70  24.20  15.07   9.77   
2     AP001  2017-11-24 19:00:00  80.00  132.00  2.10  25.18  15.15  12.02   
3     AP001  2017-11-24 20:00:00  81.50  133.25  1.95  16.25  10.23  11.58   
4     AP001  2017-11-24 21:00:00  75.25  116.00  1.43  17.48  10.43  12.03   

    CO    SO2      O3  Benzene  Toluene  Xylene  AQI AQI_Bucket  
0  0.1  11.85  126.40      0.1     6.10    0.10  NaN        NaN  
1  0.1  13.17  117.12      0.1     6.25    0.15  NaN        NaN  
2  0.1  12.08   98.98      0.2     5.98    0.18  NaN        NaN  
3  0.1  10.47  112.20      0.2     6.72    0.10  NaN        NaN  
4  0.1   9.12  106.35      0.2     5.75    0.08  NaN        NaN  


## 3. Initial Data Overview

In [3]:
print("Columns:")
print(df_station_hour.columns.tolist())

print("\nData types:")
print(df_station_hour.dtypes)

print("\nAQI_Bucket distribution:")
print(df_station_hour["AQI_Bucket"].value_counts(dropna=False))

Columns:
['StationId', 'Datetime', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI', 'AQI_Bucket']

Data types:
StationId         str
Datetime          str
PM2.5         float64
PM10          float64
NO            float64
NO2           float64
NOx           float64
NH3           float64
CO            float64
SO2           float64
O3            float64
Benzene       float64
Toluene       float64
Xylene        float64
AQI           float64
AQI_Bucket        str
dtype: object

AQI_Bucket distribution:
AQI_Bucket
Moderate        675008
NaN             570190
Satisfactory    530164
Very Poor       301150
Poor            239990
Good            152113
Severe          120468
Name: count, dtype: int64


## 4. Overall Missing Values

In [4]:
print("Missing values:")
print(df_station_hour.isna().sum())

Missing values:
StationId           0
Datetime            0
PM2.5          647689
PM10          1119252
NO             553711
NO2            528973
NOx            490808
NH3           1236618
CO             499302
SO2            742737
O3             725973
Benzene        861579
Toluene       1042366
Xylene        2075104
AQI            570190
AQI_Bucket     570190
dtype: int64


We have 2.59 million hourly station records, and missingness is substantial. For example:  

* PM2.5: ~648k missing
* PM10: ~1.12M missing
* NH3: ~1.24M missing
* Xylene: ~2.08M missing
* AQI: ~570k missing  

So we should not simply drop every row containing a missing value — that would throw away a large portion of our historical data.

## 5. AQI Pollutant Missingness

We define the seven AQI-relevant pollutants and examine their missingness in detail.

In [5]:
aqi_pollutants = [
    "PM2.5",
    "PM10",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3"
]

# 1. Overall missing values for AQI pollutants
print("Missing values in AQI pollutants:")
print(df_station_hour[aqi_pollutants].isna().sum())

# 2. Missing percentage (sorted descending)
print("\nMissing percentage:")
print(
    (df_station_hour[aqi_pollutants].isna().mean() * 100)
    .round(2)
    .sort_values(ascending=False)
)

# 3. How many pollutants are missing in each row?
missing_count_per_row = df_station_hour[aqi_pollutants].isna().sum(axis=1)

print("\nNumber of missing AQI pollutants per row:")
print(missing_count_per_row.value_counts().sort_index())

Missing values in AQI pollutants:
PM2.5     647689
PM10     1119252
NOx       490808
NH3      1236618
CO        499302
SO2       742737
O3        725973
dtype: int64

Missing percentage:
NH3      47.76
PM10     43.23
SO2      28.69
O3       28.04
PM2.5    25.02
CO       19.28
NOx      18.96
dtype: float64

Number of missing AQI pollutants per row:
0    989978
1    209743
2    608068
3    211189
4     83477
5    138039
6     61293
7    287296
Name: count, dtype: int64


1. Missingness is substantial and uneven.

Among the AQI pollutants:

* NH3 is the worst: 47.76% missing
* PM10: 43.23%
* SO2: 28.69%
* O3: 28.04%
* PM2.5: 25.02%
* CO: 19.28%
* NOx: 18.96%

So we definitely cannot use simple dropna().

2. Rows differ greatly in how much information they contain.

0 missing pollutants → 989,978 rows  
1 missing            → 209,743  
2 missing            → 608,068  
3 missing            → 211,189  
4 missing            → 83,477  
5 missing            → 138,039  
6 missing            → 61,293  
7 missing            → 287,296  

The 989,978 complete rows are valuable. But there are also 287,296 rows where all seven AQI pollutants are missing. Those completely empty AQI rows are the ones we need to treat most carefully.

So 989,978 rows (~38.2%) have complete AQI-pollutant data, while 287,296 rows (~11.1%) have none of the seven pollutants.

## 6. Time Coverage & Structural Validation

In [6]:
# Convert Datetime from text to actual datetime
df_station_hour["Datetime"] = pd.to_datetime(df_station_hour["Datetime"])

print("Overall start:", df_station_hour["Datetime"].min())
print("Overall end:", df_station_hour["Datetime"].max())

print("\nDate range by station (first 20):")
print(
    df_station_hour.groupby("StationId")["Datetime"]
    .agg(["min", "max"])
    .head(20)
)

Overall start: 2015-01-01 01:00:00
Overall end: 2020-07-01 00:00:00

Date range by station (first 20):
                          min        max
StationId                               
AP001     2017-11-24 17:00:00 2020-07-01
AP005     2016-07-01 12:00:00 2020-07-01
AS001     2019-02-16 23:00:00 2020-07-01
BR005     2020-02-27 16:00:00 2020-07-01
BR006     2020-03-03 18:00:00 2020-07-01
BR007     2015-06-01 15:00:00 2020-07-01
BR008     2019-12-24 18:00:00 2020-07-01
BR009     2019-12-24 18:00:00 2020-07-01
BR010     2019-12-24 18:00:00 2020-07-01
CH001     2019-09-02 10:00:00 2020-07-01
DL001     2018-11-11 15:00:00 2020-07-01
DL002     2015-04-04 11:00:00 2020-07-01
DL003     2018-02-01 09:00:00 2020-07-01
DL004     2017-09-07 12:00:00 2020-07-01
DL005     2018-07-04 19:00:00 2020-07-01
DL006     2017-08-31 01:00:00 2020-01-01
DL007     2015-01-01 01:00:00 2020-07-01
DL008     2015-01-01 01:00:00 2020-07-01
DL009     2018-02-01 09:00:00 2020-07-01
DL010     2018-02-01 08:00:00 2020-0

So there is no single common observation period across stations. Some stations start in 2015, while others only begin in 2019 or 2020.  

So we should not force every station onto the overall 2015–2020 timeline or treat missing observations before a station's start date as outages.

In [7]:
# --------------------------------------------------
# 1. Check duplicate station-hour records
# --------------------------------------------------

duplicate_count = df_station_hour.duplicated(
    subset=["StationId", "Datetime"]
).sum()

print("Duplicate station-hour rows:", duplicate_count)

# --------------------------------------------------
# 2. Check whether timestamps are hourly
# --------------------------------------------------

minutes = df_station_hour["Datetime"].dt.minute.value_counts()

print("\nTimestamp minutes:")
print(minutes)

# --------------------------------------------------
# 3. Create station-level summary
# --------------------------------------------------

station_summary = (
    df_station_hour.groupby("StationId")["Datetime"]
    .agg(
        min_datetime="min",
        max_datetime="max",
        total_rows="count",
        unique_timestamps="nunique"
    )
)

# --------------------------------------------------
# 4. Calculate expected number of hourly observations
# --------------------------------------------------

station_summary["expected_hours"] = (
    (
        station_summary["max_datetime"]
        - station_summary["min_datetime"]
    ).dt.total_seconds() / 3600
).astype(int) + 1

# --------------------------------------------------
# 5. Calculate missing hours
# --------------------------------------------------

station_summary["missing_hours"] = (
    station_summary["expected_hours"]
    - station_summary["unique_timestamps"]
)

# --------------------------------------------------
# 6. Display stations with largest gaps
# --------------------------------------------------

print("\nStations with largest number of missing hours:")

print(
    station_summary
    .sort_values("missing_hours", ascending=False)
    .head(20)
)

# --------------------------------------------------
# 7. Display basic station statistics
# --------------------------------------------------

print("\nNumber of stations:", len(station_summary))

print(
    "\nTotal missing hours across station ranges:",
    station_summary["missing_hours"].sum()
)

Duplicate station-hour rows: 0

Timestamp minutes:
Datetime
0    2589083
Name: count, dtype: int64

Stations with largest number of missing hours:
                 min_datetime max_datetime  total_rows  unique_timestamps  \
StationId                                                                   
AP001     2017-11-24 17:00:00   2020-07-01       22784              22784   
AP005     2016-07-01 12:00:00   2020-07-01       35053              35053   
AS001     2019-02-16 23:00:00   2020-07-01       12002              12002   
BR005     2020-02-27 16:00:00   2020-07-01        2985               2985   
BR006     2020-03-03 18:00:00   2020-07-01        2863               2863   
BR007     2015-06-01 15:00:00   2020-07-01       44554              44554   
BR008     2019-12-24 18:00:00   2020-07-01        4543               4543   
BR009     2019-12-24 18:00:00   2020-07-01        4543               4543   
BR010     2019-12-24 18:00:00   2020-07-01        4543               4543   
CH001 

1. No duplicate station-hour records

Duplicate station-hour rows: 0  
So each StationId + Datetime combination is unique.  

2. All timestamps are exactly on the hour

0  |  2,589,083  
Every timestamp has minute = 0. So the dataset really is hourly.  

3. No missing timestamps within each station's observed period

Total missing hours across station ranges: 0  

It means the earlier long "missing blocks" we found are not missing timestamps. They are rows where the timestamp exists but the pollutant measurements are all NaN.

Timestamp exists  
     ↓
Station has an hourly record  
     ↓
Some/all pollutant values may be missing  

4. 110 Unique stations are actually present in df_station_hour.csv for the 26 cities

Number of stations: 110  

5. The earlier conclusion about different station start dates still stands. We should evaluate pollutant missingness within each station's actual observation period, rather than assuming all stations should cover 2015–2020.
One important correction to our mental model

Previously we were calling those 8,144-hour periods "missing hours." That's technically misleading.  
They are better described as: **continuous periods where the station has hourly records, but the AQI-relevant pollutant measurements are all missing.**

## 7. Completely Missing AQI Pollutant Rows

In [8]:
# Identify rows where all seven AQI pollutants are missing
all_missing = df_station_hour[aqi_pollutants].isna().all(axis=1)

print("Total rows:", len(df_station_hour))
print("Rows where all AQI pollutants are missing:", all_missing.sum())
print("Rows where at least one AQI pollutant is available:", (~all_missing).sum())

# See which stations contribute the most completely-missing rows
missing_rows = df_station_hour[all_missing]

print("\nCompletely missing pollutant rows by station:")
print(missing_rows["StationId"].value_counts().head(20))

Total rows: 2589083
Rows where all AQI pollutants are missing: 287296
Rows where at least one AQI pollutant is available: 2301787

Completely missing pollutant rows by station:
StationId
DL007    19082
GJ001    15716
DL011    15142
DL008    15021
DL014    10383
KA005    10122
BR007     9711
DL013     9287
DL002     8850
JH001     8368
DL021     8165
DL028     6985
KA010     6729
KA003     6463
TG006     5722
TN001     5721
TG005     5576
KA009     4951
OD001     4858
HR014     4615
Name: count, dtype: int64


### DL007 Gap Inspection

DL007 has the highest count of completely-missing rows. We inspect its timestamps to understand the gap pattern.

In [9]:
# Inspect DL007, since it has the most completely missing records.

missing_dl007 = df_station_hour[
    (df_station_hour["StationId"] == "DL007") & all_missing
].copy()

print("Number of completely missing rows:", len(missing_dl007))

print("\nFirst 20 missing timestamps:")
print(missing_dl007["Datetime"].head(20).to_string(index=False))

print("\nLast 20 missing timestamps:")
print(missing_dl007["Datetime"].tail(20).to_string(index=False))

Number of completely missing rows: 19082

First 20 missing timestamps:
2015-03-30 12:00:00
2015-03-30 13:00:00
2015-03-30 14:00:00
2015-05-01 07:00:00
2015-05-01 08:00:00
2015-05-01 09:00:00
2015-05-01 10:00:00
2015-05-01 11:00:00
2015-05-01 12:00:00
2015-05-01 13:00:00
2015-05-01 14:00:00
2015-05-01 15:00:00
2015-05-01 16:00:00
2015-05-01 17:00:00
2015-05-01 18:00:00
2015-05-01 19:00:00
2015-05-01 20:00:00
2015-05-01 21:00:00
2015-05-01 22:00:00
2015-05-01 23:00:00

Last 20 missing timestamps:
2019-03-23 02:00:00
2019-03-23 03:00:00
2019-03-23 04:00:00
2019-03-23 05:00:00
2019-03-23 06:00:00
2019-03-23 07:00:00
2019-03-23 08:00:00
2019-03-23 09:00:00
2019-03-23 10:00:00
2019-03-23 11:00:00
2019-03-23 12:00:00
2019-03-23 13:00:00
2019-03-23 14:00:00
2019-03-23 15:00:00
2019-03-23 16:00:00
2019-03-23 17:00:00
2019-04-08 14:00:00
2019-04-08 15:00:00
2019-04-08 16:00:00
2019-04-08 17:00:00


The missing rows are clearly occurring in blocks, not just as isolated missing cells.

For example, DL007 has:  
* a small 3-hour gap on March 30, 2015
* a much longer continuous gap beginning May 1, 2015
* another long gap around March 23, 2019
* another gap on April 8, 2019

This confirms that we should treat these as time-series gaps/outages, not ordinary missing values that we blindly interpolate.

In [10]:
# Measure the length of the missing blocks in DL007.

# Sort timestamps
missing_dl007 = missing_dl007.sort_values("Datetime")

# Difference between consecutive missing timestamps
time_difference = missing_dl007["Datetime"].diff()

# A new gap starts whenever the difference is not exactly 1 hour
new_gap = time_difference != pd.Timedelta(hours=1)

# Give each continuous block a group number
gap_group = new_gap.cumsum()

gap_lengths = (
    missing_dl007.groupby(gap_group)
    .size()
    .sort_values(ascending=False)
)

print("Number of continuous missing blocks:", len(gap_lengths))

print("\nLargest 10 missing blocks (hours):")
print(gap_lengths.head(10))

Number of continuous missing blocks: 46

Largest 10 missing blocks (hours):
Datetime
6     8144
8     5794
2     2946
5      714
7      607
36     162
10     119
30      71
11      52
24      48
dtype: int64


For DL007:  

* 46 separate missing blocks
* Largest block: 8,144 consecutive hours ≈ 339 days
* Second largest: 5,794 hours ≈ 241 days
* Third largest: 2,946 hours ≈ 123 days

These are clearly long station outages/data gaps, not values that we should interpolate across. Interpolating a 339-day gap would create artificial pollution data.  

So our eventual cleaning approach should distinguish between:  

* short missing gaps → potentially suitable for time-series imputation
* long station outages → leave unavailable / exclude those periods from modelling

## 8. Station-Level Pollutant Missingness

In [11]:
# Station-level missing percentage for each AQI pollutant
station_missing = (
    df_station_hour.groupby("StationId")[aqi_pollutants]
    .apply(lambda x: x.isna().mean() * 100)
)

station_missing["Average_Missing_%"] = station_missing.mean(axis=1)

print("Stations with highest average pollutant missingness:")
print(
    station_missing
    .sort_values("Average_Missing_%", ascending=False)
    .head(20)
)

# --------------------------------------------------
# Completely missing AQI pollutant rows by station
# --------------------------------------------------

complete_missing_by_station = (
    df_station_hour.loc[all_missing]
    .groupby("StationId")
    .size()
    .sort_values(ascending=False)
)

print("\nStations with most completely missing AQI rows:")
print(complete_missing_by_station.head(20))

Stations with highest average pollutant missingness:
                PM2.5        PM10         NOx         NH3          CO  \
StationId                                                               
DL011      100.000000  100.000000  100.000000  100.000000  100.000000   
KA005      100.000000   34.349718   49.243859  100.000000   25.401431   
KA010      100.000000   39.473684   17.063531  100.000000   16.756677   
DL007       42.496680   42.229001   40.550299  100.000000   39.614459   
MH005       67.193725   69.407786   30.212483  100.000000    6.227175   
DL013       40.033616  100.000000   38.869522   38.923473   26.489874   
JH001       71.190009   35.504014  100.000000   40.071365   54.083854   
DL008       53.981989   61.012201   32.604997   49.780046   60.601345   
DL033       49.109811  100.000000    7.179615  100.000000   24.062085   
GJ001       35.406292   81.472029   32.885126  100.000000   34.507802   
KA003       23.300548  100.000000   35.258964  100.000000   49.416916  

1. Missingness is strongly station-dependent.

For example:

DL011 → 83.25% average missingness  
KA005 → 63.32%  
KA010 → 58.95%  
DL007 → 58.18%  
MH005 → 57.81%  

This means we should not assume one global imputation strategy will work equally well for every station.

2. Some individual pollutants are completely unavailable at particular stations.

For example:

DL011:  
PM2.5 → 100% missing  
PM10  → 100%  
NOx   → 100%  
NH3   → 100%  
CO    → 100%  

Similarly, several stations have 100% missing NH3, and some have 100% missing PM10. This is an important warning: we cannot promise to reconstruct every pollutant for every station.

## 9. Gap Structure Analysis (Per-Station)

We identify whether pollutant missing values occur in short gaps or long continuous gaps, **grouped by station** so that runs don't bleed across station boundaries.

In [12]:
# Sort by station and time so within-station runs are contiguous
df_station_hour = df_station_hour.sort_values(["StationId", "Datetime"])

# Find continuous missing runs for each pollutant, grouped per station
results = []

for pollutant in aqi_pollutants:

    missing = df_station_hour[pollutant].isna()

    # A new run starts whenever missing/not-missing status changes within a station
    run_id = (
        missing != missing.groupby(df_station_hour["StationId"]).shift()
    ).cumsum()

    runs = (
        df_station_hour[missing]
        .groupby(["StationId", run_id[missing]])
        .size()
    )

    if len(runs) > 0:
        results.append({
            "Pollutant": pollutant,
            "Total_missing_rows": missing.sum(),
            "Longest_missing_run_hours": runs.max(),
            "Runs_1_to_6_hours": (runs <= 6).sum(),
            "Runs_7_to_24_hours": ((runs > 6) & (runs <= 24)).sum(),
            "Runs_over_24_hours": (runs > 24).sum(),
        })

result_df = pd.DataFrame(results)

print("\nMissing-run analysis (per-station grouped):")
print(result_df.to_string(index=False))


Missing-run analysis (per-station grouped):
Pollutant  Total_missing_rows  Longest_missing_run_hours  Runs_1_to_6_hours  Runs_7_to_24_hours  Runs_over_24_hours
    PM2.5              647689                      40736              19184                3920                1661
     PM10             1119252                      48192              14571                2820                1255
      NOx              490808                      37236              14832                3871                1565
      NH3             1236618                      48192              10742                2557                1147
       CO              499302                      37236              24440                6363                2220
      SO2              742737                      48192              25958                3769                1690
       O3              725973                      48192              16073                3332                1568


This confirms something important: the missingness is not mainly short gaps.

Every pollutant has some very long continuous missing runs:

Pollutant	Longest missing run  
PM2.5	40,736 hours ≈ 4.6 years  
PM10	48,192 hours ≈ 5.5 years  
NOx	37,236 hours ≈ 4.3 years  
NH3	48,192 hours ≈ 5.5 years  
CO	37,236 hours ≈ 4.3 years  
SO2	48,192 hours ≈ 5.5 years  
O3	48,192 hours ≈ 5.5 years  

That tells us we absolutely should not interpolate all missing values.

There are also many shorter gaps:  
1–6 hours → potentially reasonable candidates for interpolation  
7–24 hours → need more caution  
24 hours → generally shouldn't be blindly interpolated  

But those huge runs may partly represent pollutants that were never measured at a particular station, rather than a temporary sensor outage.  

For example, we already saw stations where a pollutant is 100% missing.  

So before choosing an imputation rule, we need to distinguish:  
```
Pollutant missing  
       │
       ├── Station never measured it
       │        → don't impute
       │
       └── Station normally measures it
                │
                ├── short gap → potentially interpolate
                └── long gap  → don't blindly interpolate
```

## 10. Pollutant Availability by Station

In [13]:
# Percentage of available (non-missing) values for each pollutant at each station
availability = (
    df_station_hour
    .groupby("StationId")[aqi_pollutants]
    .apply(lambda x: x.notna().mean() * 100)
)

print("Pollutant availability (%) by station:")
print(availability.round(2))

Pollutant availability (%) by station:
           PM2.5   PM10    NOx    NH3     CO    SO2     O3
StationId                                                 
AP001      84.41  85.24  85.73  85.17  81.41  82.87  85.21
AP005      74.62  75.31  90.75  77.11  90.59  73.87  76.75
AS001      96.30  97.01  98.75  98.74  98.75  98.76  98.73
BR005      19.73  59.10  78.22  30.35  78.99  60.20  59.43
BR006      86.03  83.62  96.12  57.42  94.24  90.29  89.14
...          ...    ...    ...    ...    ...    ...    ...
WB009      97.08  97.03  96.96  96.45  96.86  98.21  97.86
WB010      95.63  95.51  96.06  96.20  97.18  97.07  96.51
WB011      87.37  87.52  87.38  87.46  88.89  87.73  88.18
WB012      85.94  91.69  96.84  96.56  98.57  98.23  98.66
WB013      89.00  88.68  94.19  95.46  97.47  89.49  90.96

[110 rows x 7 columns]


This confirms something important: availability varies substantially by station and pollutant. For example, BR005 has only 19.73% PM2.5 availability, while AS001 has ~96–99% availability across pollutants.

In [14]:
# List only station-pollutant combinations with 0% availability

zero_availability = (
    availability
    .stack()
    .loc[lambda s: s == 0]
)

print(zero_availability)

print("\nNumber of zero-availability pairs:", len(zero_availability))

StationId      
BR007      PM10    0.0
           NH3     0.0
BR010      SO2     0.0
DL004      NH3     0.0
           SO2     0.0
                  ... 
UP014      NH3     0.0
UP015      PM10    0.0
           NH3     0.0
UP016      PM10    0.0
           O3      0.0
Length: 68, dtype: float64

Number of zero-availability pairs: 68


* 110 stations
* 7 AQI pollutants
* 68 station–pollutant combinations were never measured
Therefore, for those 68 combinations, we should leave the values as missing rather than impute them.

## 11. AQI Column Analysis

In [15]:
# How many rows already have a valid AQI value?

aqi_missing = df_station_hour["AQI"].isna().sum()
aqi_available = df_station_hour["AQI"].notna().sum()

print("AQI available:", aqi_available)
print("AQI missing:", aqi_missing)
print("AQI missing percentage:", round(aqi_missing / len(df_station_hour) * 100, 2), "%")

AQI available: 2018893
AQI missing: 570190
AQI missing percentage: 22.02 %


In [16]:
# Check the AQI value range — detect obviously invalid AQI values

print("Minimum AQI:", df_station_hour["AQI"].min())
print("Maximum AQI:", df_station_hour["AQI"].max())

print("\nAQI summary:")
print(df_station_hour["AQI"].describe())

Minimum AQI: 5.0
Maximum AQI: 3133.0

AQI summary:
count    2.018893e+06
mean     1.801730e+02
std      1.404095e+02
min      5.000000e+00
25%      8.400000e+01
50%      1.310000e+02
75%      2.590000e+02
max      3.133000e+03
Name: AQI, dtype: float64


In [17]:
# Inspect the highest AQI values and their pollutant concentrations

columns_to_check = [
    "StationId",
    "Datetime",
    "AQI",
    "AQI_Bucket",
    "PM2.5",
    "PM10",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3"
]

highest_aqi = (
    df_station_hour
    .nlargest(10, "AQI")[columns_to_check]
)

print(highest_aqi.to_string(index=False))

StationId            Datetime    AQI AQI_Bucket  PM2.5  PM10    NOx  NH3     CO    SO2    O3
    GJ001 2017-11-13 00:00:00 3133.0     Severe 182.80   NaN 388.27  NaN 498.57    NaN 11.14
    GJ001 2017-11-13 01:00:00 3133.0     Severe 198.70   NaN 269.21  NaN 306.52 179.62  9.47
    GJ001 2017-11-13 02:00:00 3133.0     Severe 190.30   NaN 115.87  NaN  53.16  91.12  6.26
    GJ001 2017-11-13 03:00:00 3133.0     Severe 151.70   NaN  66.82  NaN  11.07 146.49 13.19
    GJ001 2017-11-13 04:00:00 3133.0     Severe 125.90   NaN  53.40  NaN   7.10  95.12 21.70
    GJ001 2017-11-13 05:00:00 3133.0     Severe 112.20   NaN  68.95  NaN  18.00 101.28  8.86
    GJ001 2017-11-13 06:00:00 3133.0     Severe 102.50   NaN  43.56  NaN   5.42 159.37 26.70
    GJ001 2017-11-13 07:00:00 3133.0     Severe  99.80   NaN  42.96  NaN   5.61 138.19 45.10
    GJ001 2018-12-22 22:00:00 3111.0     Severe 216.36   NaN 311.00  NaN 494.90  55.34  3.85
    GJ001 2018-12-22 23:00:00 3111.0     Severe 262.85   NaN 402.04  N

## 12. Summary & Key Conclusions

| Finding | Detail |
|---------|--------|
| No duplicate station-hour records | Each (StationId, Datetime) pair is unique |
| All timestamps are on the hour | minute == 0 for all 2.59M rows |
| No missing timestamps within a station's period | "Missing" rows have NaN measurements, not absent timestamps |
| 110 unique stations, staggered start dates | Cannot assume a uniform 2015–2020 window for all |
| NH3 (47.76%) and PM10 (43.23%) are worst-missing pollutants | Cannot use dropna() |
| 287,296 rows (~11.1%) have all seven AQI pollutants missing | Many are long contiguous outage blocks |
| Longest per-station gap: up to 48,192 hours (≈5.5 years) | Blanket interpolation is inappropriate |
| 68 station–pollutant pairs with 0% availability | These 68 combinations should never be imputed |
| AQI max = 3,133 (GJ001) | Plausible for severe industrial pollution events |

### Recommended imputation strategy (for next notebook)

```
For each station–pollutant pair:
  If availability == 0%  →  leave as NaN (never measured)
  Else if gap length ≤ 6h  →  linear interpolation within station
  Else if gap length 7–24h  →  cautious interpolation or flag
  Else (gap > 24h)          →  leave as NaN (outage period)
```

# AQI Calculation Eligibility

In [18]:
# For each station-hour, does the available data meet the conditions required to calculate a valid CPCB AQI?

# Check how many AQI-relevant pollutants are available in each row

available_aqi_pollutants = (
    df_station_hour[aqi_pollutants].notna().sum(axis=1)
)

print(available_aqi_pollutants.value_counts().sort_index())

0    287296
1     61293
2    138039
3     83477
4    211189
5    608068
6    209743
7    989978
Name: count, dtype: int64


989,978 rows have all 7 AQI-relevant pollutants available.  
And 287,296 rows have none of the 7 available.

In [19]:
# Compare AQI availability with pollutant availability

aqi_available_by_pollutants = (
    pd.DataFrame({
        "Available_Pollutants": available_aqi_pollutants,
        "AQI_Available": df_station_hour["AQI"].notna()
    })
    .groupby("Available_Pollutants")["AQI_Available"]
    .agg(["count", "sum"])
)

aqi_available_by_pollutants["AQI_Available_Percentage"] = (
    aqi_available_by_pollutants["sum"]
    / aqi_available_by_pollutants["count"]
    * 100
).round(2)

print(aqi_available_by_pollutants)

                       count     sum  AQI_Available_Percentage
Available_Pollutants                                          
0                     287296   33037                     11.50
1                      61293    1199                      1.96
2                     138039    4470                      3.24
3                      83477   55424                     66.39
4                     211189  173901                     82.34
5                     608068  581056                     95.56
6                     209743  201228                     95.94
7                     989978  968578                     97.84


## Calculate AQI for small portion of dataset

In [20]:
# Select one station for AQI calculation verification

test_station = (
    df_station_hour[df_station_hour["StationId"] == "GJ001"]
    .copy()
)

test_station["Datetime"] = pd.to_datetime(test_station["Datetime"])

print("Rows:", len(test_station))
print("Date range:", test_station["Datetime"].min(), "to", test_station["Datetime"].max())

Rows: 48192
Date range: 2015-01-01 01:00:00 to 2020-07-01 00:00:00


In [ ]:
# Inspect AQI-related data for GJ001

columns_to_check = [
    "Datetime",
    "PM2.5",
    "PM10",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "AQI",
    "AQI_Bucket"
]

           Datetime  PM2.5  PM10   NOx  NH3   CO    SO2     O3  AQI AQI_Bucket
2015-01-01 01:00:00    NaN   NaN 36.37  NaN 1.00 122.07    NaN  NaN        NaN
2015-01-01 02:00:00    NaN   NaN 19.73  NaN 0.02  85.90    NaN  NaN        NaN
2015-01-01 03:00:00    NaN   NaN 11.08  NaN 0.08  52.83    NaN  NaN        NaN
2015-01-01 04:00:00    NaN   NaN  9.20  NaN 0.30  39.53 153.58  NaN        NaN
2015-01-01 05:00:00    NaN   NaN  7.85  NaN 0.12  32.63    NaN  NaN        NaN
2015-01-01 06:00:00    NaN   NaN 10.82  NaN 0.33  29.87  64.25  NaN        NaN
2015-01-01 07:00:00    NaN   NaN 12.47  NaN 0.45  27.41 191.96  NaN        NaN
2015-01-01 08:00:00    NaN   NaN 16.48  NaN 1.03  20.92 177.21  NaN        NaN
2015-01-01 09:00:00    NaN   NaN 18.02  NaN 1.47  16.45 122.08  NaN        NaN
2015-01-01 10:00:00    NaN   NaN 16.08  NaN 2.05  15.14    NaN  NaN        NaN


In [22]:
# Show the first 10 GJ001 observations where the original AQI is available

aqi_available_sample = (
    test_station[test_station["AQI"].notna()]
    [columns_to_check]
    .head(10)
)

print(aqi_available_sample.to_string(index=False))

           Datetime  PM2.5  PM10   NOx  NH3   CO   SO2     O3   AQI AQI_Bucket
2015-01-29 09:00:00  80.47   NaN 20.65  NaN 1.42 32.26  46.71 288.0       Poor
2015-01-29 10:00:00  87.80   NaN 22.22  NaN 2.03 78.95  21.88 181.0   Moderate
2015-01-29 11:00:00  90.53   NaN 22.12  NaN 1.72 95.86  67.67 182.0   Moderate
2015-01-29 12:00:00  83.53   NaN 16.48  NaN 0.75 44.91  88.92 182.0   Moderate
2015-01-29 13:00:00  68.40   NaN 12.50  NaN 0.58 34.57 116.75 179.0   Moderate
2015-01-29 14:00:00  67.57   NaN 15.12  NaN 0.63 27.26  82.96 176.0   Moderate
2015-01-29 15:00:00  61.13   NaN 12.90  NaN 0.43 34.47 106.88 173.0   Moderate
2015-01-29 16:00:00  63.43   NaN 10.83  NaN 0.45 40.32  82.54 171.0   Moderate
2015-01-29 17:00:00  54.37   NaN 13.38  NaN 0.72 35.77  73.54 167.0   Moderate
2015-01-29 18:00:00  49.20   NaN 22.13  NaN 0.53 27.72 143.96 169.0   Moderate


In [23]:
df_station_hour["Datetime"] = pd.to_datetime(df_station_hour["Datetime"])

is_sorted = (
    df_station_hour
    .sort_values(["StationId", "Datetime"])
    .index
    .equals(df_station_hour.index)
)

print("Correctly sorted:", is_sorted)

Correctly sorted: True


In [24]:
# Create rolling averages

df_station_hour["PM10_24hr_avg"] = (
    df_station_hour.groupby("StationId")["PM10"]
    .rolling(window=24, min_periods=16)
    .mean()
    .values
)

df_station_hour["PM2.5_24hr_avg"] = (
    df_station_hour.groupby("StationId")["PM2.5"]
    .rolling(window=24, min_periods=16)
    .mean()
    .values
)

df_station_hour["SO2_24hr_avg"] = (
    df_station_hour.groupby("StationId")["SO2"]
    .rolling(window=24, min_periods=16)
    .mean()
    .values
)

df_station_hour["NOx_24hr_avg"] = (
    df_station_hour.groupby("StationId")["NOx"]
    .rolling(window=24, min_periods=16)
    .mean()
    .values
)

df_station_hour["NH3_24hr_avg"] = (
    df_station_hour.groupby("StationId")["NH3"]
    .rolling(window=24, min_periods=16)
    .mean()
    .values
)

df_station_hour["CO_8hr_max"] = (
    df_station_hour.groupby("StationId")["CO"]
    .rolling(window=8, min_periods=1)
    .max()
    .values
)

df_station_hour["O3_8hr_max"] = (
    df_station_hour.groupby("StationId")["O3"]
    .rolling(window=8, min_periods=1)
    .max()
    .values
)

print(
    df_station_hour[
        ["Datetime", "PM2.5_24hr_avg", "PM10_24hr_avg",
         "CO_8hr_max", "O3_8hr_max"]
    ].head(30)
)

              Datetime  PM2.5_24hr_avg  PM10_24hr_avg  CO_8hr_max  O3_8hr_max
0  2017-11-24 17:00:00             NaN            NaN         0.1      126.40
1  2017-11-24 18:00:00             NaN            NaN         0.1      126.40
2  2017-11-24 19:00:00             NaN            NaN         0.1      126.40
3  2017-11-24 20:00:00             NaN            NaN         0.1      126.40
4  2017-11-24 21:00:00             NaN            NaN         0.1      126.40
5  2017-11-24 22:00:00             NaN            NaN         0.1      126.40
6  2017-11-24 23:00:00             NaN            NaN         0.1      126.40
7  2017-11-25 00:00:00             NaN            NaN         0.1      126.40
8  2017-11-25 01:00:00             NaN            NaN         0.3      121.50
9  2017-11-25 02:00:00             NaN            NaN         0.3      121.50
10 2017-11-25 03:00:00             NaN            NaN         0.3      121.50
11 2017-11-25 04:00:00             NaN            NaN         0.

In [26]:
# Sub-Index

def get_PM25_subindex(x):
    if x <= 30:
        return x * 50 / 30
    elif x <= 60:
        return 50 + (x - 30) * 50 / 30
    elif x <= 90:
        return 100 + (x - 60) * 100 / 30
    elif x <= 120:
        return 200 + (x - 90) * 100 / 30
    elif x <= 250:
        return 300 + (x - 120) * 100 / 130
    elif x > 250:
        return 400 + (x - 250) * 100 / 130
    else:
        return 0


def get_PM10_subindex(x):
    if x <= 50:
        return x
    elif x <= 100:
        return x
    elif x <= 250:
        return 100 + (x - 100) * 100 / 150
    elif x <= 350:
        return 200 + (x - 250)
    elif x <= 430:
        return 300 + (x - 350) * 100 / 80
    elif x > 430:
        return 400 + (x - 430) * 100 / 80
    else:
        return 0


def get_SO2_subindex(x):
    if x <= 40:
        return x * 50 / 40
    elif x <= 80:
        return 50 + (x - 40) * 50 / 40
    elif x <= 380:
        return 100 + (x - 80) * 100 / 300
    elif x <= 800:
        return 200 + (x - 380) * 100 / 420
    elif x <= 1600:
        return 300 + (x - 800) * 100 / 800
    elif x > 1600:
        return 400 + (x - 1600) * 100 / 800
    else:
        return 0


def get_NOx_subindex(x):
    if x <= 40:
        return x * 50 / 40
    elif x <= 80:
        return 50 + (x - 40) * 50 / 40
    elif x <= 180:
        return 100 + (x - 80) * 100 / 100
    elif x <= 280:
        return 200 + (x - 180) * 100 / 100
    elif x <= 400:
        return 300 + (x - 280) * 100 / 120
    elif x > 400:
        return 400 + (x - 400) * 100 / 120
    else:
        return 0


def get_NH3_subindex(x):
    if x <= 200:
        return x * 50 / 200
    elif x <= 400:
        return 50 + (x - 200) * 50 / 200
    elif x <= 800:
        return 100 + (x - 400) * 100 / 400
    elif x <= 1200:
        return 200 + (x - 800) * 100 / 400
    elif x <= 1800:
        return 300 + (x - 1200) * 100 / 600
    elif x > 1800:
        return 400 + (x - 1800) * 100 / 600
    else:
        return 0


def get_CO_subindex(x):
    if x <= 1:
        return x * 50 / 1
    elif x <= 2:
        return 50 + (x - 1) * 50 / 1
    elif x <= 10:
        return 100 + (x - 2) * 100 / 8
    elif x <= 17:
        return 200 + (x - 10) * 100 / 7
    elif x <= 34:
        return 300 + (x - 17) * 100 / 17
    elif x > 34:
        return 400 + (x - 34) * 100 / 17
    else:
        return 0


def get_O3_subindex(x):
    if x <= 50:
        return x * 50 / 50
    elif x <= 100:
        return 50 + (x - 50) * 50 / 50
    elif x <= 168:
        return 100 + (x - 100) * 100 / 68
    elif x <= 208:
        return 200 + (x - 168) * 100 / 40
    elif x <= 748:
        return 300 + (x - 208) * 100 / 539
    elif x > 748:
        return 400 + (x - 400) * 100 / 539
    else:
        return 0


df_station_hour["PM10_SubIndex"] = df_station_hour["PM10_24hr_avg"].apply(get_PM10_subindex)
df_station_hour["SO2_SubIndex"] = df_station_hour["SO2_24hr_avg"].apply(get_SO2_subindex)
df_station_hour["NOx_SubIndex"] = df_station_hour["NOx_24hr_avg"].apply(get_NOx_subindex)
df_station_hour["NH3_SubIndex"] = df_station_hour["NH3_24hr_avg"].apply(get_NH3_subindex)
df_station_hour["CO_SubIndex"] = df_station_hour["CO_8hr_max"].apply(get_CO_subindex)
df_station_hour["O3_SubIndex"] = df_station_hour["O3_8hr_max"].apply(get_O3_subindex)

print(
    df_station_hour[
        [
            "Datetime",
            "PM2.5_SubIndex",
            "PM10_SubIndex",
            "SO2_SubIndex",
            "NOx_SubIndex",
            "NH3_SubIndex",
            "CO_SubIndex",
            "O3_SubIndex"
        ]
    ].head(30)
)

              Datetime  PM2.5_SubIndex  PM10_SubIndex  SO2_SubIndex  \
0  2017-11-24 17:00:00        0.000000       0.000000      0.000000   
1  2017-11-24 18:00:00        0.000000       0.000000      0.000000   
2  2017-11-24 19:00:00        0.000000       0.000000      0.000000   
3  2017-11-24 20:00:00        0.000000       0.000000      0.000000   
4  2017-11-24 21:00:00        0.000000       0.000000      0.000000   
5  2017-11-24 22:00:00        0.000000       0.000000      0.000000   
6  2017-11-24 23:00:00        0.000000       0.000000      0.000000   
7  2017-11-25 00:00:00        0.000000       0.000000      0.000000   
8  2017-11-25 01:00:00        0.000000       0.000000      0.000000   
9  2017-11-25 02:00:00        0.000000       0.000000      0.000000   
10 2017-11-25 03:00:00        0.000000       0.000000      0.000000   
11 2017-11-25 04:00:00        0.000000       0.000000      0.000000   
12 2017-11-25 05:00:00        0.000000       0.000000      0.000000   
13 201

In [28]:
subindex_columns = [
    "PM2.5_SubIndex",
    "PM10_SubIndex",
    "SO2_SubIndex",
    "NOx_SubIndex",
    "NH3_SubIndex",
    "CO_SubIndex",
    "O3_SubIndex"
]

# Count how many pollutant sub-indices are available
df_station_hour["Checks"] = (
    df_station_hour[subindex_columns] > 0
).sum(axis=1)

# Final AQI = highest pollutant sub-index
df_station_hour["AQI_calculated"] = (
    df_station_hour[subindex_columns]
    .max(axis=1)
    .round()
)

# PM2.5 or PM10 must be available
df_station_hour.loc[
    df_station_hour["PM2.5_SubIndex"] + df_station_hour["PM10_SubIndex"] <= 0,
    "AQI_calculated"
] = np.nan

# At least 3 pollutants must be available
df_station_hour.loc[
    df_station_hour["Checks"] < 3,
    "AQI_calculated"
] = np.nan

print(
    df_station_hour[
        ["StationId", "Datetime", "AQI", "AQI_calculated", "Checks"]
    ]
    .dropna(subset=["AQI_calculated"])
    .head(20)
)

   StationId            Datetime    AQI  AQI_calculated  Checks
16     AP001 2017-11-25 09:00:00  155.0           155.0       7
17     AP001 2017-11-25 10:00:00  159.0           159.0       7
18     AP001 2017-11-25 11:00:00  173.0           173.0       7
19     AP001 2017-11-25 12:00:00  184.0           184.0       7
20     AP001 2017-11-25 13:00:00  184.0           184.0       7
21     AP001 2017-11-25 14:00:00  191.0           191.0       7
22     AP001 2017-11-25 15:00:00  191.0           191.0       7
23     AP001 2017-11-25 16:00:00  191.0           191.0       7
24     AP001 2017-11-25 17:00:00  191.0           191.0       7
25     AP001 2017-11-25 18:00:00  191.0           191.0       7
26     AP001 2017-11-25 19:00:00  191.0           191.0       7
27     AP001 2017-11-25 20:00:00  191.0           191.0       7
28     AP001 2017-11-25 21:00:00  191.0           191.0       7
29     AP001 2017-11-25 22:00:00  191.0           191.0       7
30     AP001 2017-11-25 23:00:00  186.0 

In [ ]:
# Validation: measure how often our calculation matches the dataset's existing AQI.

comparison = df_station_hour[
    df_station_hour["AQI"].notna() &
    df_station_hour["AQI_calculated"].notna()
].copy()

comparison["AQI_Match"] = (
    comparison["AQI"] == comparison["AQI_calculated"]
)

print("Rows compared:", len(comparison))
print("Matching rows:", comparison["AQI_Match"].sum())
print("Mismatching rows:", (~comparison["AQI_Match"]).sum())

print(
    "Match percentage:",
    round(comparison["AQI_Match"].mean() * 100, 2),
    "%"
)

Rows compared: 2017972
Matching rows: 2016725
Mismatching rows: 1247
Match percentage: 99.94 %


In [30]:
# Show large the differences actually are

mismatches = comparison[
    comparison["AQI"] != comparison["AQI_calculated"]
].copy()

mismatches["AQI_Difference"] = (
    mismatches["AQI_calculated"] - mismatches["AQI"]
)

print("Number of mismatches:", len(mismatches))

print("\nDifference summary:")
print(mismatches["AQI_Difference"].describe())

print("\nMost common differences:")
print(mismatches["AQI_Difference"].value_counts().head(10))

Number of mismatches: 1247

Difference summary:
count    1247.000000
mean        0.236568
std         0.972005
min        -1.000000
25%        -1.000000
50%         1.000000
75%         1.000000
max         1.000000
Name: AQI_Difference, dtype: float64

Most common differences:
AQI_Difference
 1.0    771
-1.0    476
Name: count, dtype: int64


In [31]:
# How many rows have missing original AQI but a valid AQI_calculated?

aqi_recoverable = df_station_hour[
    df_station_hour["AQI"].isna() &
    df_station_hour["AQI_calculated"].notna()
]

print("Original AQI missing:", df_station_hour["AQI"].isna().sum())
print("AQI values we can calculate:", len(aqi_recoverable))

Original AQI missing: 570190
AQI values we can calculate: 0


In [32]:
# How many rows have a valid calculated AQI overall.

print("Calculated AQI available:", df_station_hour["AQI_calculated"].notna().sum())
print("Calculated AQI missing:", df_station_hour["AQI_calculated"].isna().sum())

Calculated AQI available: 2017972
Calculated AQI missing: 571111
